####Working with numbers
1. Using mathematical expressions
2. [Using mathematical functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#mathematical-functions)
3. [Using aggregate functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html#aggregate-functions)

####1. Load invoices data to a dataframe

In [0]:
retail_df = (
    spark.read.format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/invoices.csv")
)

retail_df.show(10)

####2. Calculate total_value = Quantity * unit Price for each invoice line item


2.1 Simple approach of creating expressions

In [0]:
from pyspark.sql.functions import expr
retail_df.withColumn("Total Value", expr("quantity * unitprice")).display()

2.2 Using column expressions

In [0]:
from pyspark.sql.functions import col,round

retail_df.withColumn("Total Value", round(col("quantity") * col("unitprice"),2)).limit(10).display(10)


2.3 Using expression variable - 1


In [0]:
total_value_expr = round(col("quantity") * col("unitprice"),2) #we can use expression and use it in withColumn
retail_df.withColumn("Total Value", total_value_expr).limit(3).display()

2.4 Using expression variable - 2


In [0]:
total_value_expr_2 = expr("round(quantity * unitprice, 2)")

retail_df.withColumn("total_value", total_value_expr_2).limit(3).display()

####3. Perform the following exploratory analysis on invoices data
1. Can we make invoice numbers a numeric field?
2. Analyize quantity to identify potentially invalid records
3. Analyze unit price to identify potentially invalid records


3.1 Analyze using dataframe summary

Available statistics are:
```
  count - mean - stddev - min - max - approximate percentiles
```

In [0]:
#retail_df.describe(['InvoiceNo', 'Quantity', 'UnitPrice']).display() # No we can't make invoice numbers a numeric because there are no numeric entries present inside it
retail_df.summary().select('summary', 'InvoiceNo', 'Quantity', 'UnitPrice').display()

3.2 Using sql functions

In [0]:
from pyspark.sql.functions import min, max, percentile

retail_df.select(min("unitprice"), expr("max(unitprice)"), percentile("unitprice", 0.99)).display()